In [2]:
!python -V

Python 3.12.3


In [4]:
import pandas as pd

In [5]:
import pickle

In [6]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [7]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1778961188764, experiment_id='1', last_update_time=1778961188764, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [8]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [9]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [10]:
df_train = df_train[:1000]
df_val = df_val[:1000]

In [11]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [12]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [13]:
import xgboost as xgb
from pathlib import Path

In [14]:
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [15]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=30,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/opt/conda/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [19:58:16] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:10.49062
[1]	validation-rmse:9.92174
[2]	validation-rmse:9.42562
[3]	validation-rmse:8.99620
[4]	validation-rmse:8.62500
[5]	validation-rmse:8.30687
[6]	validation-rmse:8.03422
[7]	validation-rmse:7.80090
[8]	validation-rmse:7.60360
[9]	validation-rmse:7.43610
[10]	validation-rmse:7.29693
[11]	validation-rmse:7.17768
[12]	validation-rmse:7.08335
[13]	validation-rmse:7.00061
[14]	validation-rmse:6.93472
[15]	validation-rmse:6.87768
[16]	validation-rmse:6.82865
[17]	validation-rmse:6.79233
[18]	validation-rmse:6.76116
[19]	validation-rmse:6.73665
[20]	validation-rmse:6.71670
[21]	validation-rmse:6.69759
[22]	validation-rmse:6.68209
[23]	validation-rmse:6.66984
[24]	validation-rmse:6.66083
[25]	validation-rmse:6.65299
[26]	validation-rmse:6.64834
[27]	validation-rmse:6.64773
[28]	validation-rmse:6.64046
[29]	validation-rmse:6.64645


2026/05/16 19:58:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run adorable-cub-77 at: http://localhost:5000/#/experiments/1/runs/30c700f770414fca96e615e5b3881074
🧪 View experiment at: http://localhost:5000/#/experiments/1
